# Tinjau query tambahan
Notebook ini mengekstrak PAA yang sudah tersimpan dan membaca keputusan seleksi. Tidak melakukan pencarian SerpApi atau panggilan Gemini. Daftar query batch utama pertama tetap dipertahankan.
Seleksi awal dilakukan asisten, bukan penilai manusia independen. Periksa alasan pada `needs_review` dan `excluded`. Setelah respons utama bertambah, menjalankan ulang notebook bisa menambahkan kandidat `pending` yang belum ditinjau.

In [3]:
import sys
from pathlib import Path
from collections import Counter
from html import escape
from IPython.display import HTML, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/prepare_query_expansion.py').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from prepare_query_expansion import prepare, read_json, read_rows
config = read_json(ROOT / 'configs/query_expansion_01.json')
OUTPUT = ROOT / 'data/interim/query_expansion' / config['expansion_id']

def show(rows, columns, limit=100):
    head = ''.join('<th>' + escape(c) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(r.get(c, ''))) + '</td>' for c in columns) + '</tr>' for r in rows[:limit])
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))
    print(f'Ditampilkan {min(limit, len(rows))} dari {len(rows)} baris.')

In [4]:
# Jalankan ulang untuk memuat PAA tersimpan terbaru; keputusan lama tetap dibaca.
candidates = prepare(config)
print('Diterima per domain:', dict(Counter(r['domain'] for r in candidates if r['selection_status'] == 'accepted')))
DOMAIN = ''  # kosong untuk semua; atau kesehatan / keuangan / teknologi
STATUS = 'needs_review'  # accepted / needs_review / excluded / pending / kosong
selected = [r for r in candidates if (not DOMAIN or r['domain'] == DOMAIN) and (not STATUS or r['selection_status'] == STATUS)]
show(selected, ['query_text', 'domain', 'selection_status', 'selection_reason', 'retrieval_query', 'source_batch', 'paa_depth'])

Kemunculan PAA: 260 | query unik dengan domain: 252
Kandidat baru: 245 {'kesehatan': 93, 'keuangan': 85, 'teknologi': 67}
Status: {'accepted': 123, 'needs_review': 29, 'excluded': 31, 'pending': 62}
Output: D:\Kuliah\TA\final-assignment\data\interim\query_expansion\query_expansion_01
Diterima per domain: {'kesehatan': 62, 'keuangan': 30, 'teknologi': 31}


query_text,domain,selection_status,selection_reason,retrieval_query,source_batch,paa_depth
Apakah anemia bisa menyebabkan jantung?,kesehatan,needs_review,Objek kondisi jantung tidak lengkap; teks asli perlu penilaian intent tanpa menebak penyakitnya.,jantung,paa_expansion_01,1
5 bank BUMN apa saja?,keuangan,needs_review,Jumlah lima merupakan premis pertanyaan yang perlu diperiksa tanpa mengubah teks.,bank indonesia,paa_expansion_01,1
Cara mengecek apakah kita dapat bantuan BPJS Ketenagakerjaan?,keuangan,needs_review,Jenis program bantuan belum disebutkan.,bpjs ketenagakerjaan,paa_expansion_01,1
BPJS Ketenagakerjaan dicairkan berapa?,keuangan,needs_review,Program manfaat dan kondisi kepesertaan tidak disebutkan.,bpjs ketenagakerjaan,paa_expansion_01,1
KUR BRI 2026 pinjaman 50 juta angsuran berapa?,keuangan,needs_review,Tenor dan skema pinjaman belum tersedia.,kur bri 2026,paa_expansion_01,1
Kenapa harga saham anjlok?,keuangan,needs_review,Tidak jelas apakah meminta penjelasan umum atau suatu peristiwa; emiten/periode belum disebutkan.,ihsg,paa_expansion_01,1
1 lot saham harga berapa?,keuangan,needs_review,Emiten dan periode harga belum disebutkan; perlu memastikan intent edukasi lot atau harga aktual.,ihsg,paa_expansion_01,1
1 lot saham bisa untung berapa?,keuangan,needs_review,"Emiten, horizon, dan jenis keuntungan belum disebutkan.",investor,paa_expansion_01,1
Siapa investor nomor 1 di Indonesia?,keuangan,needs_review,Kriteria nomor satu tidak disebutkan.,investor,paa_expansion_01,1
Investasi 1 juta per bulan dapat berapa?,keuangan,needs_review,"Instrumen, jangka waktu, dan asumsi imbal hasil belum disebutkan.",investor,paa_expansion_01,1


Ditampilkan 29 dari 29 baris.


## Koreksi keputusan
Isi dictionary hanya untuk pertanyaan yang ingin kamu ubah, dengan teks persis dari tabel. `accepted` berarti query diterima untuk pengumpulan berikutnya; bukan bukti lolos grounding atau artikel siap pakai. Bahasa `id` wajib untuk penerimaan.
Keputusan disimpan di `data/manual/query_expansion_01_decisions.csv`. Teks yang sama pada domain berbeda perlu diperiksa lewat `query_id`; notebook akan memberi error jika pilihan berdasarkan teks ambigu.

In [ ]:
decisions_by_text = {
    # 'Teks persis pertanyaan': {
    #     'language': 'id', 'status': 'accepted',
    #     'reason': 'Alasan keputusan setelah saya tinjau.'
    # },
}
changes = {}
for text, decision in decisions_by_text.items():
    matches = [r for r in candidates if r['query_text'] == text]
    if len(matches) != 1:
        raise ValueError(f'Teks tidak ditemukan atau ambigu: {text}')
    changes[matches[0]['query_id']] = {**decision, 'reviewer': 'peneliti'}
if changes:
    candidates = prepare(config, changes)
else:
    print('Tidak ada koreksi; keputusan tersimpan dipertahankan.')

In [ ]:
accepted = read_rows(OUTPUT / 'accepted_new.csv')
print('Query tambahan diterima:', len(accepted))
print('Berkas untuk batch berikutnya:', OUTPUT / 'accepted_new.csv')
show(accepted, ['query_id', 'query_text', 'domain', 'topic_id', 'source_batch', 'parent_query_id'], limit=30)